# Oracle DB Query Notebook
連線至 Fubon Oracle 資料庫並執行 SQL 查詢

## 1. DB Connect

In [ ]:
import oracledb
import pandas as pd
import time

oracledb.init_oracle_client(
    lib_dir=r"C:\Users\ian.leong\Downloads\instantclient-basic-windows.x64-23.9.0.25.07\instantclient_23_9"
)

# ===== 請調整以下參數 =====
HOST         = "sndmp-scan.fbs100.campus"
PORT         = "5211"
SERVICE_NAME = "SNDMP_USER_DS"
ACCOUNT      = "S_IANLEONG"
PASSWORD     = "Fubon#3333"


def query_db(sql, label=""):
    """執行 SQL，回傳 DataFrame"""
    dsn = f"{HOST}:{PORT}/{SERVICE_NAME}"
    conn = oracledb.connect(user=ACCOUNT, password=PASSWORD, dsn=dsn)
    cursor = conn.cursor()
    cursor.arraysize = 10000
    t0 = time.time()
    try:
        cursor.execute(sql)
        cols = [d[0].lower() for d in cursor.description]
        data = cursor.fetchall()
        df = pd.DataFrame(data, columns=cols)
        print(f"{label} 完成：{len(df)} 筆，{len(df.columns)} 欄，耗時 {time.time()-t0:.0f}s")
        return df
    except Exception as e:
        print(f"{label} 查詢失敗：{e}")
        return None
    finally:
        cursor.close()
        conn.close()

print("環境設定完成")

環境設定完成


## 2. Testing

In [2]:
# ===== 測試 DB 連線 =====
try:
    dsn = f"{HOST}:{PORT}/{SERVICE_NAME}"
    conn = oracledb.connect(user=ACCOUNT, password=PASSWORD, dsn=dsn)
    cursor = conn.cursor()
    cursor.execute("SELECT USER, SYSDATE FROM DUAL")
    row = cursor.fetchone()
    print(f"✅ 連線成功！User: {row[0]}, Time: {row[1]}")
    cursor.close()
    conn.close()
except Exception as e:
    print(f"❌ 連線失敗：{e}")

❌ 連線失敗：ORA-28000: The account is locked.
Help: https://docs.oracle.com/error-help/db/ora-28000/


## 3. Query
替換下方 SQL 即可取得 DataFrame

In [4]:
# ✏️ 替換下方 SQL 即可
sql = """
SELECT *
FROM S_ANDYCTKUO.PARTY_NAME
WHERE ROWNUM <= 10
"""

df = query_db(sql, label="自訂查詢")
df

自訂查詢 完成：10 筆，4 欄，耗時 0s


,acct_nbr,party_id,party_id_mask,party_name
0,96000050503,22957301,22957301,富邦證券信
1,96008477777,22957301,22957301,富邦證券信
2,96008899999,22957301,22957301,富邦證券信
3,96009899011,22957301,22957301,富邦證券信
4,96009999999,22957301,22957301,富邦證券信
5,96040000070,A201632189,A261830950,陳好
6,96040000119,T101695162,T161807188,林金茂
7,96040000164,Y220343268,Y228992585,高月芬
8,96040000232,Y100031121,Y133350042,劉俊雄
9,96040000355,K120718638,K175046196,邱中鋒


## 4. Close
`query_db()` 每次查詢後會自動關閉連線，無須手動關閉

In [7]:
print("✅ query_db() 已自動管理連線，無需手動 close。")

✅ query_db() 已自動管理連線，無需手動 close。
